In [1]:
#!/bin/bash
!curl -L -o v2-plant-seedlings-dataset.zip https://www.kaggle.com/api/v1/datasets/download/vbookshelf/v2-plant-seedlings-dataset

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 3268M  100 3268M    0     0  36.0M      0  0:01:30  0:01:30 --:--:-- 35.2M


In [2]:
import zipfile

with zipfile.ZipFile('/content/v2-plant-seedlings-dataset.zip', 'r') as zip_ref:
  zip_ref.extractall('dataset')

In [3]:
!ls dataset

 Black-grass	    'Fat Hen'		 'ShepherdΓÇÖs Purse'
 Charlock	    'Loose Silky-bent'	 'Small-flowered Cranesbill'
 Cleavers	     Maize		 'Sugar beet'
'Common Chickweed'   nonsegmentedv2
'Common wheat'	    'Scentless Mayweed'


In [4]:
import shutil

shutil.rmtree('/content/dataset/nonsegmentedv2')

In [5]:
import os

dataset = '/content/dataset'
num_classes = len(os.listdir(dataset))
num_classes

12

In [6]:
import torch.nn as nn
import torch
import torchvision
from torchvision import datasets,transforms
from torch.utils.data import DataLoader,random_split
import torch.nn.functional as F

In [7]:
transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [8]:
dataset = datasets.ImageFolder(root=dataset, transform=transform)

In [9]:
torch.manual_seed(42)
n_total=len(dataset)
n_train=int(n_total*0.8)
n_val=int(n_total*0.1)
n_test=n_total-n_train-n_val

In [10]:
train_set,val_set,test_set=random_split(dataset, [n_train,n_val,n_test])

In [11]:
train_loader=DataLoader(train_set,batch_size=32, shuffle=True)
val_loader=DataLoader(val_set, batch_size=32)
test_loader=DataLoader(test_set, batch_size=32)

In [12]:
class InceptionBlock(nn.Module):

  def __init__(self,in_channels, c1_num,
               c3_red_num, c3_num,
               c5_red_num, c5_num,
               pool_num):
    super().__init__()

    self.c1 = nn.Conv2d(in_channels, c1_num, 1, 1, 'same')
    self.c2 = nn.Sequential(
        nn.Conv2d(in_channels, c3_red_num, 1, 1, 'same'),
        nn.ReLU(),
        nn.Conv2d(c3_red_num, c3_num, 3, 1, padding=1)
    )
    self.c3 = nn.Sequential(
        nn.Conv2d(in_channels, c5_red_num, 1, 1, 'same'),
        nn.ReLU(),
        nn.Conv2d(c5_red_num, c5_num, 5, 1, padding=2)
    )
    self.c4 = nn.Sequential(
        nn.MaxPool2d(3, 1, 1),
        nn.Conv2d(in_channels, pool_num, 1, 1, 'same')
    )

  def forward(self, X):
    out1 = F.relu(self.c1(X))
    out2 = F.relu(self.c2(X))
    out3 = F.relu(self.c3(X))
    out4 = F.relu(self.c4(X))

    return torch.concat([out1, out2, out3, out4], dim=1)

In [13]:
class InceptionModel(nn.Module):
  def __init__(self, in_channels: int = 3, num_classes:int = 10):
    super().__init__()

    self.prep = nn.Sequential(
        nn.Conv2d(in_channels, 64, 7, stride=2, padding=3),
        nn.ReLU(),
        nn.MaxPool2d(3, 2, 1),
        nn.BatchNorm2d(64),
        nn.Conv2d(64, 64, 1, padding='same'),
        nn.ReLU(),
        nn.Conv2d(64, 192, 3, padding='same'),
        nn.BatchNorm2d(192),
        nn.MaxPool2d(3, 2, 1)
    )

    self.inception_3a = InceptionBlock(192, 64, 96, 128, 16, 32, 32)
    self.inception_3b = InceptionBlock(256, 128, 128, 192, 32, 96, 64)

    self.max_pool1 = nn.MaxPool2d(3, 2, 1)

    self.inception_4a  = InceptionBlock(480, 192, 96, 208, 16, 48, 64)
    self.inception_4b  = InceptionBlock(512, 160, 112, 224, 24, 64, 64)
    self.inception_4c  = InceptionBlock(512, 128, 128, 256, 24, 64, 64)
    self.inception_4d  = InceptionBlock(512, 112, 144, 288, 32, 64, 64)
    self.inception_4e  = InceptionBlock(528, 256, 160, 320, 32, 128, 128)

    self.max_pool2 = nn.MaxPool2d(3, 2, 1)

    self.inception_5a = InceptionBlock(832, 256, 160, 320, 32, 128, 128)
    self.inception_5b = InceptionBlock(832, 384, 192, 384, 48, 128, 128)

    self.avg_pool1 = nn.AdaptiveAvgPool2d((1, 1))
    self.dropout = nn.Dropout(0.4)
    self.main_output = nn.Linear(1024, num_classes)

  def forward(self, X):
    X = F.relu(self.prep(X))
    X = self.inception_3a(X)
    X = self.inception_3b(X)
    X = self.max_pool1(X)
    X = self.inception_4a(X)
    X = self.inception_4b(X)
    X = self.inception_4c(X)
    X = self.inception_4d(X)
    X = self.inception_4e(X)
    X = self.max_pool2(X)
    X = self.inception_5a(X)
    X = self.inception_5b(X)
    X = self.avg_pool1(X)
    X = torch.flatten(X, 1)
    X = self.dropout(X)
    output = self.main_output(X)

    return output

In [14]:
next(iter(train_loader))[0].shape

torch.Size([32, 3, 224, 224])

In [15]:
number_of_classes=len(dataset.classes)
number_of_classes

12

In [16]:
model = InceptionModel(3, num_classes=number_of_classes)

In [17]:
if torch.cuda.is_available():
  device = 'cuda'
elif torch.backends.mps.is_available():
  device = 'mps'
else:
  device = 'cpu'

In [18]:
device

'cuda'

In [19]:
def train(model, optimizer, criterion, train_loader, val_loader, n_epochs, device, scheduler=None, early_stop_callback=None):
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'lr': []}
    best_val_loss = float('inf')

    print(f"{'Epoch':^7} | {'Tr. Loss':^10} | {'Val Loss':^10} | {'Val Acc':^10} | {'LR':^10}")
    print("-" * 60)

    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.
        current_lr = optimizer.param_groups[0]['lr']

        for image, label in train_loader:
            image, label = image.to(device), label.to(device).long()
            pred = model(image)
            loss = criterion(pred, label)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()

        mean_train_loss = total_loss / len(train_loader)

        model.eval()
        val_loss = 0.
        correct = 0
        total = 0
        with torch.no_grad():
            for image, label in val_loader:
                image, label = image.to(device), label.to(device).long()
                pred = model(image)
                loss = criterion(pred, label)
                val_loss += loss.item()

                predicted = torch.argmax(pred, dim=1)
                correct += (predicted == label).sum().item()
                total += label.size(0)

        mean_val_loss = val_loss / len(val_loader)
        val_acc = correct / total

        if scheduler is not None:
            scheduler.step(mean_val_loss)

        print(f"{epoch+1:^7} | {mean_train_loss:^10.4f} | {mean_val_loss:^10.4f} | {val_acc:^10.4f} | {current_lr:^10.1e}")

        if mean_val_loss < best_val_loss:
            best_val_loss = mean_val_loss
            torch.save(model.state_dict(), 'best_weather_model.pth')
            print(f"--- Model Saved! (New Best Val Loss: {best_val_loss:.4f}) ---")

        history['train_loss'].append(mean_train_loss)
        history['val_loss'].append(mean_val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)

        if early_stop_callback is not None:
            early_stop_callback(mean_val_loss)
            if early_stop_callback.early_stop:
                print(f"\nEarly stopping: {epoch+1}-ci epoxada aktivləşdi "
                      f"(Val Loss {early_stop_callback.patience} epoxadır yaxşılaşmır).")
                break

    return history

In [20]:
model.train()
model.to(device)

InceptionModel(
  (prep): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (7): BatchNorm2d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (inception_3a): InceptionBlock(
    (c1): Conv2d(192, 64, kernel_size=(1, 1), stride=(1, 1), padding=same)
    (c2): Sequential(
      (0): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1), padding=same)
      (1): ReLU()
      (2): Conv2d(96, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    )
    (c3): Sequential(
      (0): Conv2d(192, 16, 

In [21]:
class EarlyStopping:
    def __init__(self, monitor="val_loss", patience=5, mode="min", min_delta=0.0):
        self.monitor = monitor
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, current_score):
        if self.best_score is None:
            self.best_score = current_score
            return

        improved = (
            current_score < (self.best_score - self.min_delta)
            if self.mode == "min"
            else current_score > (self.best_score + self.min_delta)
        )

        if improved:
            self.best_score = current_score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

In [22]:
optimizer=torch.optim.AdamW(model.parameters(),lr=5e-4)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
early_stop_callback = EarlyStopping(monitor="val_loss", patience=5, mode="min")
criterion=nn.CrossEntropyLoss()
N_EPOCHS=10

In [23]:
train(model,optimizer,criterion,train_loader,val_loader, n_epochs=N_EPOCHS,device=device,scheduler=lr_scheduler,early_stop_callback=early_stop_callback)

 Epoch  |  Tr. Loss  |  Val Loss  |  Val Acc   |     LR    
------------------------------------------------------------
   1    |   2.4306   |   2.4251   |   0.1392   |  5.0e-04  
--- Model Saved! (New Best Val Loss: 2.4251) ---
   2    |   2.1820   |   2.0304   |   0.2514   |  5.0e-04  
--- Model Saved! (New Best Val Loss: 2.0304) ---
   3    |   1.7983   |   1.8531   |   0.3870   |  5.0e-04  
--- Model Saved! (New Best Val Loss: 1.8531) ---
   4    |   1.5957   |   1.9558   |   0.3472   |  5.0e-04  
   5    |   1.4001   |   1.4171   |   0.5136   |  5.0e-04  
--- Model Saved! (New Best Val Loss: 1.4171) ---
   6    |   1.1551   |   1.0904   |   0.6401   |  5.0e-04  
--- Model Saved! (New Best Val Loss: 1.0904) ---
   7    |   1.0045   |   0.9427   |   0.6781   |  5.0e-04  
--- Model Saved! (New Best Val Loss: 0.9427) ---
   8    |   0.8892   |   0.9752   |   0.6926   |  5.0e-04  
   9    |   0.8416   |   0.8711   |   0.7143   |  5.0e-04  
--- Model Saved! (New Best Val Loss: 0.8711) 

{'train_loss': [2.430588301994818,
  2.1820306958054467,
  1.798260192219302,
  1.5957339581825751,
  1.4000738327451747,
  1.1550899516764304,
  1.0044540552784214,
  0.8892131756535537,
  0.8416457167632289,
  0.7826963437975739],
 'val_loss': [2.425081796116299,
  2.0304426153500876,
  1.8530607620875041,
  1.9558482501241896,
  1.4170935087733798,
  1.090389370918274,
  0.9426579276720682,
  0.9752321408854591,
  0.8710600237051646,
  0.9160782612032361],
 'val_acc': [0.13924050632911392,
  0.2513562386980108,
  0.38698010849909587,
  0.3471971066907776,
  0.5135623869801085,
  0.6401446654611211,
  0.6781193490054249,
  0.6925858951175407,
  0.7142857142857143,
  0.6799276672694394],
 'lr': [0.0005,
  0.0005,
  0.0005,
  0.0005,
  0.0005,
  0.0005,
  0.0005,
  0.0005,
  0.0005,
  0.0005]}